In [ ]:
import os
import pandas as pd

# =====================================================
# ROOT DATASET PATH
# =====================================================
ROOT_DIR = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD/archive/RetinalOCT_Dataset/RetinalOCT_Dataset"

SPLITS = ["train", "val", "test"]
CLASSES = ["AMD", "CNV", "CSR", "DME", "DR", "DRUSEN", "MH", "NORMAL"]

records = []

# =====================================================
# WALK THROUGH DATASET
# =====================================================
for split in SPLITS:
    split_path = os.path.join(ROOT_DIR, split)

    for label in CLASSES:
        label_path = os.path.join(split_path, label)

        if not os.path.isdir(label_path):
            continue

        for fname in os.listdir(label_path):
            if fname.lower().endswith(".jpg"):
                img_path = os.path.join(label_path, fname)

                records.append({
                    "split": split,
                    "label_text": label,
                    "new_file_path": img_path
                })

# =====================================================
# CREATE DATAFRAME
# =====================================================
df = pd.DataFrame(records)
df["binary_label"] = (df["label_text"] != "NORMAL").astype(int)

print("Total images:", len(df))

df.head()


In [ ]:
df['label_text'].value_counts()

In [ ]:
df['binary_label'].value_counts()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

# 1. Dataset Configurations
datasets = {
    "NEH": "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/neh_test_reduced.csv",
    "UCSD": "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ucsd_filtered.csv",
    "DHU": "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/dhu_filtered.csv",
    "OCT C8": "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/octC8_filtered.csv"
}

# 2. Setup Figure (1 Row, 4 Columns)
fig, axes = plt.subplots(1, 4, figsize=(20, 5), dpi=300)

# 3. Control the "Border" between samples
# wspace=0.05 creates a thin white vertical gap between the subplots
plt.subplots_adjust(wspace=0.05, hspace=0, left=0.01, right=0.99, top=0.99, bottom=0.01)

target_size = (512, 512)

# 4. Iterate and Plot
for col_idx, (name, csv_path) in enumerate(datasets.items()):
    try:
        # Load dataset and pick 1 random sample
        df = pd.read_csv(csv_path)
        sample = df.sample(1).iloc[0]
        img_path = sample['new_file_path']
        
        # Load and resize image
        img = Image.open(img_path).convert('L')
        img = img.resize(target_size, resample=Image.LANCZOS)
        
        ax = axes[col_idx]
        ax.imshow(img, cmap='gray')
        
        # Completely remove axes, ticks, and labels
        ax.axis('off')
                        
    except Exception as e:
        print(f"Error loading {name}: {e}")
        axes[col_idx].axis('off')

# Ensure layout is tight but respects the wspace defined above
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

# --------------------------------------------
# Function: Letterbox into 512×512 black canvas
# --------------------------------------------
def resize_with_black_background(img, target_size=(512, 512)):
    tw, th = target_size
    w, h = img.size

    # keep aspect ratio
    scale = min(tw / w, th / h)
    new_w = int(w * scale)
    new_h = int(h * scale)

    img_resized = img.resize((new_w, new_h), Image.LANCZOS)

    # black canvas
    canvas = Image.new('L', target_size, color=0)

    # center the resized image
    paste_x = (tw - new_w) // 2
    paste_y = (th - new_h) // 2
    canvas.paste(img_resized, (paste_x, paste_y))

    return canvas


# --------------------------------------------
# Plot
# --------------------------------------------
fig, axes = plt.subplots(3, 4, figsize=(20, 15), dpi=300)
plt.subplots_adjust(wspace=0.05, hspace=0.05)

target_size = (512, 512)
row_labels = ["Normal", "AMD", "DME"]
dataset_names = list(image_data.keys())

for col_idx, name in enumerate(dataset_names):
    paths = image_data[name]

    for row_idx, img_path in enumerate(paths):
        try:
            img = Image.open(img_path).convert('L')
            img = resize_with_black_background(img, target_size)

            ax = axes[row_idx, col_idx]
            ax.imshow(img, cmap='gray', vmin=0, vmax=255)
            ax.axis('off')

            if row_idx == 0:
                ax.set_title(name, fontsize=24, fontweight='bold', pad=25)

            if col_idx == 0:
                ax.text(-80, 256, row_labels[row_idx],
                        rotation=90, va='center', ha='right',
                        fontsize=24, fontweight='bold',
                        transform=ax.transData)

        except Exception as e:
            print(f"Error loading {img_path}: {e}")
            axes[row_idx, col_idx].axis('off')

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import os

# 1. Define specific file paths provided
# Structure: {Dataset_Name: [Path_to_Normal, Path_to_AMD_Drusen, Path_to_DME]}
image_data = {
    "NEH": [
        "/home/bharath/Downloads/sample_images_2/NEH_normal.TIFF",
        "/home/bharath/Downloads/sample_images_2/NEH_AMD.TIFF",
        "/home/bharath/Downloads/sample_images_2/NEH_abnormal_dme.TIFF"
    ],
    "UCSD": [
        "/home/bharath/Downloads/sample_images_2/UCSD_normal.jpeg",
        "/home/bharath/Downloads/sample_images_2/UCSD_abnormal_DRUSEN.jpeg",
        "/home/bharath/Downloads/sample_images_2/UCSD_abnormal_DME.jpeg"
    ],
    "DHU": [
        "/home/bharath/Downloads/sample_images_2/DHU_normal.tif",
        "/home/bharath/Downloads/sample_images_2/DHU_abnormal_AMD.tif",
        "/home/bharath/Downloads/sample_images_2/DHU_abnormal_DME.tif"
    ],
    "OCT C8": [
        # "/home/bharath/Downloads/sample_images_2/octc8_normal.jpg",
        "/home/bharath/Downloads/sample_images_2/normal_test_1182.jpg",
        "/home/bharath/Downloads/sample_images_2/octc8_abnormal_AMD.jpg",
        "/home/bharath/Downloads/sample_images_2/octc8_abnormal_DME.jpg"
    ]
}

# 2. Setup Figure (3 Rows, 4 Columns)
fig, axes = plt.subplots(3, 4, figsize=(20, 15), dpi=300)
plt.subplots_adjust(wspace=0.05, hspace=0.05)

target_size = (512, 512)
row_labels = ["Normal", "AMD", "DME"]
dataset_names = list(image_data.keys())

# 3. Iterate and Plot
for col_idx, name in enumerate(dataset_names):
    paths = image_data[name]
    
    for row_idx, img_path in enumerate(paths):
        try:
            # High-quality resize using LANCZOS for medical detail preservation
            img = Image.open(img_path).convert('L')
            img = img.resize(target_size, resample=Image.LANCZOS)
            
            ax = axes[row_idx, col_idx]
            ax.imshow(img, cmap='gray')
            ax.axis('off')
            
            # Add Dataset Names as Column Titles (Top Row Only)
            if row_idx == 0:
                ax.set_title(name, fontsize=35, fontweight='bold', pad=25)
            
            # Add Class Labels (Left-most Column Only)
            if col_idx == 0:
                # Offset label to the left of the row
                ax.text(-80, 256, row_labels[row_idx], rotation=90, 
                        va='center', ha='right', fontsize=35, 
                        fontweight='bold', transform=ax.transData)
                        
        except Exception as e:
            print(f"Error loading {img_path}: {e}")
            axes[row_idx, col_idx].axis('off')
            axes[row_idx, col_idx].text(0.5, 0.5, "File Not Found", 
                                        ha='center', va='center', fontsize=12)

# 4. Final Formatting and Save
plt.tight_layout()
# bbox_inches='tight' ensures the side labels (Normal/AMD/DME) aren't clipped
plt.savefig("OCT_MultiClass_Comparison_Matrix.png", bbox_inches='tight', pad_inches=0.5, dpi=300)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import os

# 1. Define specific file paths provided
# Structure: {Dataset_Name: [Path_to_Normal, Path_to_Abnormal]}
image_data = {
    "NEH": [
        "/home/bharath/Downloads/sample_images/NEH_normal.TIFF",
        "/home/bharath/Downloads/sample_images/NEH_abnormal_dme.TIFF"
    ],
    "UCSD": [
        "/home/bharath/Downloads/sample_images/UCSD_normal.jpeg",
        "/home/bharath/Downloads/sample_images/UCSD_abnormal_cnv.jpeg"
    ],
    "DHU": [
        "/home/bharath/Downloads/sample_images/dhu_normal.tif",
        "/home/bharath/Downloads/sample_images/dhu_abnormal_amd.tif"
    ],
    "OCT C8": [
        "/home/bharath/Downloads/sample_images/octc8_normal.jpg", # As provided in your list
        "/home/bharath/Downloads/sample_images/octc8_abnormal_dme.jpg"
    ]
}

# 2. Setup Figure (2 Rows, 4 Columns)
fig, axes = plt.subplots(2, 4, figsize=(20, 10), dpi=300)
plt.subplots_adjust(wspace=0.03, hspace=0.03)

target_size = (512, 512)
row_labels = ["Normal", "Abnormal"]
dataset_names = list(image_data.keys())

# 3. Iterate and Plot
for col_idx, name in enumerate(dataset_names):
    paths = image_data[name]
    
    for row_idx, img_path in enumerate(paths):
        try:
            # High-quality resize using LANCZOS for medical detail preservation
            img = Image.open(img_path).convert('L')
            img = img.resize(target_size, resample=Image.LANCZOS)
            
            ax = axes[row_idx, col_idx]
            ax.imshow(img, cmap='gray')
            ax.axis('off')
            
            # Add Dataset Names as Column Titles (Top Row Only)
            if row_idx == 0:
                ax.set_title(name, fontsize=24, fontweight='bold', pad=25)
            
            # Add Class Labels (Left-most Column Only)
            if col_idx == 0:
                ax.text(-60, 256, row_labels[row_idx], rotation=90, 
                        va='center', ha='right', fontsize=24, 
                        fontweight='bold', transform=ax.transData)
                        
        except Exception as e:
            print(f"Error loading {img_path}: {e}")
            axes[row_idx, col_idx].axis('off')
            axes[row_idx, col_idx].text(0.5, 0.5, "Error", ha='center')

# 4. Final Formatting and Save
plt.tight_layout()
# bbox_inches='tight' is crucial to ensure labels aren't cut off in the saved PNG
plt.savefig("OCT_Dataset_Final_Matrix.png", bbox_inches='tight', pad_inches=0.2, dpi=300)
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import os

# 1. Dataset Configurations
datasets = {
    "NEH": "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/neh_test_reduced.csv",
    "UCSD": "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ucsd_filtered.csv",
    "DHU": "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/dhu_filtered.csv",
    "OCT C8": "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/octC8_filtered.csv"
}

# 2. Setup Figure (2 Rows, 4 Columns)
fig, axes = plt.subplots(2, 4, figsize=(20, 10), dpi=300)
plt.subplots_adjust(wspace=0.03, hspace=0.03)

target_size = (512, 512)
row_labels = {0: "Normal", 1: "Abnormal"}

# 3. Iterate and Plot
for col_idx, (name, csv_path) in enumerate(datasets.items()):
    # Load dataset
    df = pd.read_csv(csv_path)
    
    for row_idx, label in enumerate([0, 1]):
        # Select representative sample
        # Change random_state if you want to cycle through different images
        sample = df[df['binary_label'] == label].sample(1, random_state=45).iloc[0]
        img_path = sample['new_file_path']
        
        try:
            # High-quality resize
            img = Image.open(img_path).convert('L')
            img = img.resize(target_size, resample=Image.LANCZOS)
            
            ax = axes[row_idx, col_idx]
            ax.imshow(img, cmap='gray')
            ax.axis('off')
            
            # Column Titles (Top Row Only)
            if row_idx == 0:
                ax.set_title(name, fontsize=22, fontweight='bold', pad=25)
            
            # Row Labels (Left-most Column Only)
            if col_idx == 0:
                # Positioned slightly to the left of the image
                ax.text(-50, 256, row_labels[label], rotation=90, 
                        va='center', ha='right', fontsize=22, 
                        fontweight='bold', transform=ax.transData)
                        
        except Exception as e:
            print(f"Error loading {img_path}: {e}")
            axes[row_idx, col_idx].axis('off')

# 4. Save for Journal Publication
plt.tight_layout()
plt.savefig("OCT_Domain_Matrix_512_seed43.png", bbox_inches='tight', pad_inches=0.2, dpi=300)
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import os

# 1. Define dataset configurations
datasets = {
    "NEH": "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/neh_test_reduced.csv",
    "UCSD": "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ucsd_filtered.csv",
    "DHU": "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/dhu_filtered.csv",
    "OCT C8": "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/octC8_filtered.csv"
}

# 2. Setup the plot (4 rows, 2 columns)
fig, axes = plt.subplots(4, 2, figsize=(10, 16), dpi=300)
plt.subplots_adjust(wspace=0.05, hspace=0.1)

# Titles for columns
col_titles = ["Normal (Label 0)", "Abnormal (Label 1)"]

for row_idx, (name, csv_path) in enumerate(datasets.items()):
    df = pd.read_csv(csv_path)
    
    for col_idx, label in enumerate([0, 1]):
        # Get a random sample for the current label
        sample = df[df['binary_label'] == label].sample(1, random_state=42).iloc[0]
        img_path = sample['new_file_path']
        
        # Load and display image
        try:
            img = Image.open(img_path).convert('L') # Convert to grayscale for uniformity
            ax = axes[row_idx, col_idx]
            ax.imshow(img, cmap='gray')
            ax.axis('off')
            
            # Add Column Titles only for the first row
            if row_idx == 0:
                ax.set_title(col_titles[col_idx], fontsize=16, fontweight='bold', pad=15)
            
            # Add Row Labels (Dataset Names) only for the first column
            if col_idx == 0:
                ax.text(-15, img.size[1]//2, name, rotation=90, va='center', ha='right', 
                        fontsize=18, fontweight='bold')
        except Exception as e:
            print(f"Error loading {img_path}: {e}")
            axes[row_idx, col_idx].axis('off')
            axes[row_idx, col_idx].text(0.5, 0.5, "Image Not Found", ha='center')

# 3. Final layout tweaks
plt.tight_layout()
plt.savefig("dataset_samples_comparison.png", bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
import random
import matplotlib.pyplot as plt
from PIL import Image

# =====================================================
# RANDOMLY SAMPLE 4 IMAGES
# =====================================================
sample_df = df.sample(n=4, random_state=None)

# =====================================================
# PLOT
# =====================================================
plt.figure(figsize=(12, 6))

for idx, (_, row) in enumerate(sample_df.iterrows()):
    img_path = row["new_file_path"]
    split = row["split"]
    label = row["label_text"]

    # Read image
    img = Image.open(img_path).convert("L")  # OCTs are grayscale

    # Plot
    plt.subplot(1, 4, idx + 1)
    plt.imshow(img, cmap="gray")
    plt.title(f"{label}\n({split})", fontsize=10)
    plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
df["shape"] = df["new_file_path"].apply(lambda p: f"{Image.open(p).size[1]}x{Image.open(p).size[0]}")


In [ ]:
import pandas as pd

df = pd.read_csv("/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/octC8_filtered.csv")
df['shape'].value_counts()

In [ ]:
df['shape'].value_counts()

In [ ]:
label_shape_counts = df.groupby(["label_text", "shape"]).size().reset_index(name="count")
label_shape_counts = (
    df.groupby(["label_text", "shape"])
      .size()
      .reset_index(name="count")
      .sort_values(["label_text", "count"], ascending=[True, False])
)

print(label_shape_counts)


In [ ]:
df.to_csv("octC8.csv", index=False)

In [ ]:
filtered_df = df[
    (
        (df["label_text"] == "NORMAL") &
        (df["shape"].isin(["496x512", "512x512", "496x768"]))
    )
    |
    (
        (df["label_text"] == "AMD") &
        (df["shape"] == "512x1000")
    )
    |
    (
        (df["label_text"] == "DME") &
        (df["shape"].isin(["496x512", "512x512", "496x768"]))
    )
].reset_index(drop=True)

print(len(filtered_df))


In [ ]:
label_shape_counts = filtered_df.groupby(["label_text", "shape"]).size().reset_index(name="count")
label_shape_counts = (
    filtered_df.groupby(["label_text", "shape"])
      .size()
      .reset_index(name="count")
      .sort_values(["label_text", "count"], ascending=[True, False])
)

print(label_shape_counts)

In [ ]:
# =====================================================
# FIXED RANDOM SEED
# =====================================================
RANDOM_STATE = 42

# =====================================================
# SAMPLE AMD
# =====================================================
amd_df = (
    filtered_df[filtered_df["label_text"] == "AMD"]
    .sample(n=3000, random_state=RANDOM_STATE)
)

# =====================================================
# SAMPLE DME
# =====================================================
dme_df = (
    filtered_df[filtered_df["label_text"] == "DME"]
    .sample(n=2881, random_state=RANDOM_STATE)
)

# =====================================================
# KEEP ALL NORMAL
# =====================================================
normal_df = filtered_df[filtered_df["label_text"] == "NORMAL"]

# =====================================================
# COMBINE FINAL DATAFRAME
# =====================================================
final_df = (
    pd.concat([amd_df, dme_df, normal_df], axis=0)
    .reset_index(drop=True)
)

print("Final dataset size:", len(final_df))
# Class counts
print(final_df["label_text"].value_counts())

# Binary balance
print(final_df["binary_label"].value_counts())


In [ ]:
final_df.head()

In [ ]:
# final_df.to_csv("octC8_filtered_imbalanced.csv", index=False)

In [ ]:
final_df['binary_label'].value_counts()